In [1]:
# Importing the pandas library for data manipulation and analysis
import pandas as pd
# Importing the numpy library for numerical operations
import numpy as np
# Suppressing warning messages (useful for cleaner outputs during analysis)
import warnings
warnings.filterwarnings('ignore')  # This line disables warning messages that might pop up from pandas/numpy, etc.

In [2]:
# Reading a CSV file from the specified path and loading it into a pandas DataFrame called 'df'
# 'r' before the string indicates a raw string, which treats backslashes as literal characters (avoiding escape sequences)
df = pd.read_csv(r"C:\Users\LENOVO\Documents\PostGrad\BSc Hons\4ZOL509_Research\data\seive_percentage.csv")

In [3]:
def compute_all_methods(row, sieve_columns, phi_sizes, mm_sizes):
    """
    Compute sediment statistics using:
    - Method of Moments (Arithmetic, Geometric, Logarithmic)
    - Folk & Ward (phi and mm)
    """

    weights = row[sieve_columns].values.astype(float)
    weights = weights / weights.sum()  # Normalize to proportions

    # === Method of Moments ===
    # Arithmetic (mm)
    mean_arith = np.sum(weights * mm_sizes)
    sorting_arith = np.sqrt(np.sum(weights * (mm_sizes - mean_arith)**2))
    skew_arith = np.sum(weights * (mm_sizes - mean_arith)**3) / (sorting_arith**3)
    kurt_arith = np.sum(weights * (mm_sizes - mean_arith)**4) / (sorting_arith**4)

    # Geometric (mm)
    mm_sizes_pos = np.where(mm_sizes == 0, 0.01, mm_sizes)  # avoid log(0)
    log_mm = np.log(mm_sizes_pos)
    mean_geom = np.exp(np.sum(weights * log_mm))
    sorting_geom = np.sqrt(np.sum(weights * (log_mm - np.log(mean_geom))**2))
    skew_geom = np.sum(weights * (log_mm - np.log(mean_geom))**3) / (sorting_geom**3)
    kurt_geom = np.sum(weights * (log_mm - np.log(mean_geom))**4) / (sorting_geom**4)

    # Logarithmic (phi)
    mean_phi = np.sum(weights * phi_sizes)
    sorting_phi = np.sqrt(np.sum(weights * (phi_sizes - mean_phi)**2))
    skew_phi = np.sum(weights * (phi_sizes - mean_phi)**3) / (sorting_phi**3)
    kurt_phi = np.sum(weights * (phi_sizes - mean_phi)**4) / (sorting_phi**4)

    # === Folk & Ward (phi) ===
    cum_freq = np.cumsum(weights) * 100
    percentiles = [5, 16, 25, 50, 75, 84, 95]
    phi_interp = np.interp(percentiles, cum_freq, phi_sizes)
    phi5, phi16, phi25, phi50, phi75, phi84, phi95 = phi_interp

    mean_fw_phi = (phi16 + phi50 + phi84) / 3
    sorting_fw_phi = ((phi84 - phi16) / 4) + ((phi95 - phi5) / 6.6)
    skewness_fw_phi = (((phi16 + phi84 - 2 * phi50) / (2 * (phi84 - phi16))) +
                       ((phi5 + phi95 - 2 * phi50) / (2 * (phi95 - phi5))))
    kurtosis_fw_phi = (phi95 - phi5) / (2.44 * (phi75 - phi25))

    # Convert mean phi to mm
    mean_fw_mm = 2 ** (-mean_fw_phi)

    # Description based on mean and sorting (simplified)
    if mean_fw_phi < 0:
        desc = "Very coarse sand"
    elif 0 <= mean_fw_phi < 1:
        desc = "Coarse sand"
    elif 1 <= mean_fw_phi < 2:
        desc = "Medium sand"
    elif 2 <= mean_fw_phi < 3:
        desc = "Fine sand"
    elif 3 <= mean_fw_phi < 4:
        desc = "Very fine sand"
    else:
        desc = "Silt or finer"

    return pd.Series({
        # Method of Moments (Arithmetic)
        "mean_arith_mm": mean_arith,
        "sorting_arith_mm": sorting_arith,
        "skewness_arith_mm": skew_arith,
        "kurtosis_arith_mm": kurt_arith,

        # Method of Moments (Geometric)
        "mean_geom_mm": mean_geom,
        "sorting_geom_mm": sorting_geom,
        "skewness_geom_mm": skew_geom,
        "kurtosis_geom_mm": kurt_geom,

        # Method of Moments (Logarithmic)
        "mean_log_phi": mean_phi,
        "sorting_log_phi": sorting_phi,
        "skewness_log_phi": skew_phi,
        "kurtosis_log_phi": kurt_phi,

        # Folk & Ward Method
        "mean_fw_phi": mean_fw_phi,
        "mean_fw_mm": mean_fw_mm,
        "sorting_fw_phi": sorting_fw_phi,
        "skewness_fw_phi": skewness_fw_phi,
        "kurtosis_fw_phi": kurtosis_fw_phi,
        "fw_description": desc
    })


In [4]:
sieve_columns = [
    'sieve_2um_perc',     # 2000 µm = 2.0 mm
    'sieve_1um_perc',     # 1000 µm = 1.0 mm
    'sieve_500um_perc',   # 500 µm = 0.5 mm
    'sieve_250um_perc',   # 250 µm = 0.25 mm
    'sieve_125um_perc',   # 125 µm = 0.125 mm
    'sieve_63um_perc',    # 63 µm = 0.063 mm
    'sieve_pan_perc'      # Finer than 63 µm (let’s assume 0.031 mm)
]

# Grain sizes in mm (must match the same order)
mm_sizes = np.array([2.0, 1.0, 0.5, 0.25, 0.125, 0.063, 0.031])
phi_sizes = -np.log2(mm_sizes)


In [5]:
result = df.apply(lambda row: compute_all_methods(row, sieve_columns, phi_sizes, mm_sizes), axis=1)

# # Combine the original DataFrame with the results
results = pd.concat([df[['site','replicate']], result], axis=1)

In [6]:
# Save the 'df_perc' DataFrame to a CSV file named 'seive_percentage.csv'
# index=False ensures that the row index is not written to the CSV file
results.to_csv(r"C:\Users\LENOVO\Documents\PostGrad\BSc Hons\4ZOL509_Research\data\gradistat.csv", index=False)

# Print a message to confirm that the file was saved successfully
print("CSV file saved successfully!")

CSV file saved successfully!
